# rlmflow coding-agent walkthrough

Build a recursive coding agent with rlmflow, run a task, and inspect what happened.

This notebook mirrors `examples/coding-agent/agent.py` — the same workspace, runtime, tools, and `RLMFlow` config — but as a notebook so you can poke at every step.

By default it loads the **saved boids-simulation trace** under `examples/coding-agent/workspace/trace/` so the visualization cells run offline. Flip `RUN_LIVE = True` in the run cell to actually execute the agent (needs an OpenAI key, and a Docker image if you want sandboxed execution).

## 1. Build the agent

Four pieces snap together:

- `Workspace` — durable, branchable storage for sessions, traces, and any files the agent writes.
- `Runtime` — where REPL code runs. `LocalRuntime` runs in-process; `DockerRuntime` runs each step in a container.
- Tools — plain Python functions registered on the runtime. `FILE_TOOLS` gives the agent `read_file`, `write_file`, `edit_file`, `ls`, `grep`, etc.
- `RLMFlow` — wires an LLM client (and optional cheaper alternates) to the runtime + workspace, and exposes `start` / `step` / `fork`.

In [1]:
from pathlib import Path

from rlmflow.llm import OpenAIClient
from rlmflow.rlm import RLMConfig, RLMFlow
from rlmflow.runtime.local import LocalRuntime
from rlmflow.tools import FILE_TOOLS
from rlmflow.workspace import Workspace


def build_agent(workspace_dir: str | Path = "./notebook-coding-agent", max_depth: int = 2, max_iterations: int = 30) -> RLMFlow:
    """Construct a coding agent identical to examples/coding-agent/agent.py."""
    workspace = Workspace.create(Path(workspace_dir).resolve())
    runtime = LocalRuntime(workspace=workspace)
    runtime.register_tools(FILE_TOOLS)

    return RLMFlow(
        llm_client=OpenAIClient("gpt-5"),
        runtime=runtime,
        workspace=workspace,
        config=RLMConfig(max_depth=max_depth, max_iterations=max_iterations),
        llm_clients={
            "fast": {
                "model": OpenAIClient("gpt-5-mini"),
                "description": "Cheap model for smaller subtasks",
            },
        },
    )

workspace_path = "./notebook-coding-agent"

## 2. Run a task

`agent.start(query)` returns the initial `Node`. Every call to `agent.step(node)` then advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a new immutable `Node`. That sequence of nodes *is* the run.

`live_view()` is a context-manager that opens a self-updating Rich tree; you own the loop and just hand it the latest state on each tick. The agent loop and the visualization are decoupled — swap `live_view()` for any other display, or drop it entirely with no change to the loop.

In [48]:
from rlmflow.utils.viz import live_view

TASK = """Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add configurations -- just the canvas. The speed should be a constant, not based on size. Split each component into separate files and delegate the work. Make sure to run and test the components after they are done. The main runnable interface should be in index.html. Save in output/boids-simulation."""

agent = build_agent(max_depth=2)
state = agent.start(TASK)
states = [state]

In [49]:
print(agent.build_system_prompt(state))

## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with fresh context windows.

## REPL

- Every response MUST contain exactly one ```repl``` code block.
- Tools are already in the REPL namespace — call them directly.
- Variables persist across turns.
- `AGENT_ID`, `DEPTH`, `MAX_DEPTH` are set. You cannot delegate when `DEPTH == MAX_DEPTH`.
- Call `done(answer)` when finished.
- **Decompose programmatically** — when a task has independent parts, write code that creates those subtasks, delegates them, waits for results, and combines the answers.
- **Prefer parallel delegation for independent work** — build a list of handles and call `yield wait(*handles)` once instead of doing independent subtasks sequentially.
- **Explore before solving** — when files, long context, unknown data schemas, or tool outputs matter, first inspect samples, lengths, keys, or representative lines before fi

In [50]:
from rlmflow.utils.viz import live_view

with live_view() as view:
    view(state)
    while not state.finished:
        state = agent.step(state)
        states.append(state)
        view(state)

print(f"{len(states)} states  ·  final: {state.agent_id} [{state.type}]")
print(f"query : {states[0].query[:120]!r}...")
print(f"result: {(state.result or '')[:200]}")

Output()

4 states  ·  final: root [result]
query : 'Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add '...
result: Boids simulation created at output/boids-simulation with index.html, style.css, and script.js. Contract and basic checks passed. Open index.html to run.


In [51]:
from rlmflow.utils.trace import save_trace

trace_path = save_trace(states, agent.workspace.trace_dir, metadata={"task": TASK})
print(f"trace -> {trace_path}  ({len(states)} states)")

trace -> /Users/shyam/Code/rlmkit/examples/notebooks/notebook-coding-agent/trace/trace.json  (4 states)


In [57]:
import html as _html
import re
from IPython.display import HTML

# Build a self-contained page from the agent's index.html, then embed via
# <iframe srcdoc=...> (Jupyter doesn't resolve relative iframe URLs).
# Handles BOTH ES modules (recursive flatten + strip import/export) and plain
# <script src=...> tags, so it doesn't matter which style the agent picked.
# candidates = sorted(agent.workspace.root.glob("output/**/index.html"))
# if not candidates:
#     raise FileNotFoundError(f"no index.html under {agent.workspace.root}")
# html_path = candidates[-1]
# Offline fallback — flip these two lines to render the bundled saved run instead
# of whatever the agent just produced (handy when running without an API key):
# from pathlib import Path
html_path = Path("../data/notebook-coding-agent/output/boids-simulation/index.html")
# html_path = Path("../data/notebook-coding-agent/output/boids-simulation/index.html")
base = html_path.parent


def _inline_css(m):
    p = base / m.group(1)
    return f"<style>\n{p.read_text()}\n</style>" if p.exists() else m.group(0)


def _inline_plain_js(m):
    p = base / m.group(1)
    return f"<script>\n{p.read_text()}\n</script>" if p.exists() else m.group(0)


def _flatten_module(entry: str) -> str:
    """DFS-flatten an ES-module graph into one inline <script>, deps first."""
    visited: set = set()
    chunks: list[str] = []

    def visit(path):
        path = path.resolve()
        if path in visited or not path.exists():
            return
        visited.add(path)
        src = path.read_text()
        for m in re.finditer(
            r'^\s*import\b.*?\bfrom\s+[\'"]([^\'"]+)[\'"]', src, flags=re.MULTILINE
        ):
            visit(path.parent / m.group(1))
        src = re.sub(
            r'^\s*import\b.*?\bfrom\s+[\'"][^\'"]+[\'"];?\s*$',
            "", src, flags=re.MULTILINE,
        )
        src = re.sub(r"^\s*export\s+default\s+", "", src, flags=re.MULTILINE)
        src = re.sub(r"^\s*export\s+", "", src, flags=re.MULTILINE)
        chunks.append(f"// === {path.name} ===\n{src}")

    visit(base / entry)
    return "<script>\n" + "\n".join(chunks) + "\n</script>"


content = html_path.read_text()
content = re.sub(r'<link\b[^>]*\bhref="([^"]+)"[^>]*/?>', _inline_css, content, flags=re.I)
# ES-module <script type="module" src=...> first (recursive), then any plain <script src=>.
content = re.sub(
    r'<script\b(?=[^>]*\btype="module")[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    lambda m: _flatten_module(m.group(1)), content, flags=re.I,
)
content = re.sub(
    r'<script\b[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    _inline_plain_js, content, flags=re.I,
)

print(f"rendering {html_path}")
HTML(
    f'<iframe srcdoc="{_html.escape(content, quote=True)}" '
    f'width="100%" height="600" '
    f'style="border:1px solid #ddd; border-radius:6px; background:#0a0a0f"></iframe>'
)

rendering ../data/notebook-coding-agent/output/boids-simulation/index.html


/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning:

Consider using IPython.display.IFrame instead



## View the tree

In [55]:
from rlmflow.utils.viewer import open_viewer

# Pass `session=` so the viewer can reconstruct the full per-agent chat
# (system / user / assistant / tool / result), not just the snapshot tree.
# `states` alone gives you only what was captured in the trace.
open_viewer(
    states,
    session="../data/notebook-coding-agent/session",
    inline=True,
    height=720,
    quiet=True,
)

## 3. Inspect the final tree

Returning a `Node` from a Jupyter cell renders it as an HTML tree (via `_repr_html_`). For a complete picture, pick the snapshot with the most nodes — early states only show what's been produced so far.

In [8]:
richest = max(states, key=lambda s: len(s.walk()))
richest

root [supervising] {default:gpt-5}
  root.index.html [query] {fast}
  root.style.css [query] {fast}
  root.utils.js [query] {fast}
  root.boids.js [query] {fast}
  root.renderer.js [query] {fast}
  root.main.js [query] {fast}

In [9]:
print(richest.tree())

root [supervising] {default:gpt-5}
  root.index.html [query] {fast}
  root.style.css [query] {fast}
  root.utils.js [query] {fast}
  root.boids.js [query] {fast}
  root.renderer.js [query] {fast}
  root.main.js [query] {fast}


## 4. Query the graph

Search lives directly on `Node` — no I/O, no rendering, just predicates over the subtree.

- `state.walk()` — all nodes in DFS order
- `state.leaves()` — nodes with no materialized children
- `state.results()` / `state.errors()` — every `ResultNode` / `ErrorNode`
- `state.find(agent_id)` — first match by agent id
- `state.where(predicate=..., **filters)` — filter by attribute or callable
- `state.path_to(node_id)` — root-to-node path

In [13]:
print(f"total nodes : {len(richest.walk())}")
print(f"leaves      : {[n.agent_id for n in richest.leaves()]}")
print(f"results     : {[n.agent_id for n in richest.results()]}")
print(f"errors      : {richest.errors() or '(none)'}")

total nodes : 7
leaves      : ['root.index.html', 'root.styles.css', 'root.main.js', 'root.flock.js', 'root.boid.js', 'root.utils.js']
results     : []
errors      : (none)


In [14]:
html = richest.find("root.html")
if html is not None:
    print(f"agent_id : {html.agent_id}")
    print(f"type     : {html.type}")
    print(f"depth    : {html.depth}")
    print(f"path_to  : {[n.agent_id for n in richest.path_to(html.id)]}")
    print(f"summary  : {(getattr(html, 'result', '') or '')[:160]}")

In [15]:
# Anything matching a predicate — here, deep results.
for n in richest.where(lambda n: n.type == "result" and n.depth > 0):
    print(f"  {n.agent_id} (depth {n.depth})")

## 5. Step-indexed timeline

`viz.gantt` prints one row per agent and one column per step, with single-letter glyphs for what each agent did at each tick (`Q`uery, `A`ction, `O`bservation, `S`upervising, `R`esume, `F`inal result, `E`rror).

In [16]:
from rlmflow.utils.viz import gantt

gantt(states)

┏━━━━━━━━━━━━━━━┳━┳━┳━┳━┳━┓
┃agent          ┃0┃1┃2┃3┃4┃
┡━━━━━━━━━━━━━━━╇━╇━╇━╇━╇━┩
│root           │Q│S│S│S│F│
│root.index.html│ │Q│F│F│F│
│root.styles.css│ │Q│F│F│F│
│root.main.js   │ │Q│F│F│F│
│root.flock.js  │ │Q│O│F│F│
│root.boid.js   │ │Q│F│F│F│
│root.utils.js  │ │Q│F│F│F│
└───────────────┴─┴─┴─┴─┴─┘

## 6. The code it actually ran

`code_log` flattens the entire run into a script-style transcript: every action's REPL block followed by its observation, ordered by step. This is the fastest way to skim what an agent *did* without reading a tree.

In [17]:
from rlmflow.utils.viz import code_log

print(code_log(states)[:4000])

# [root] supervising
# Define the shared contract and delegate per-file work, then verify assembly.

requirements = ""
contract = '''
Project: output/boids-simulation

Overall requirements:
- Plain HTML + JavaScript (ES modules), no frameworks.
- No configuration UI — just a full-window canvas running the boids simulation.
- Boids should be fast, colorful, and leave trails.
- The main runnable interface is index.html.
- Split components into separate files as listed below.
- Use only the specified filenames and relative imports exactly as written.

File responsibilities (CONTRACT):
1) index.html
   - Minimal HTML document with:
     * <link rel="stylesheet" href="styles.css"> in <head>.
     * Exactly one canvas: <canvas id="boids"></canvas> inside <body>.
     * At the end of <body>, include: <script type="module" src="main.js"></script>
   - Title: "Boids Simulation"
   - No other UI elements.

2) styles.css
   - Make the canvas fill the entire window.
   - Black background, no scrol

## 7. Topology diagram

The graph is exactly what the model produced. `to_mermaid` renders an inline state-diagram in any Markdown-aware viewer (Jupyter, GitHub, VS Code). For static diagrams there's also `to_dot` (Graphviz) and `to_d2` ([d2lang.com](https://d2lang.com)).

In [18]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(richest)}\n```")

```mermaid
stateDiagram-v2
    state "root (supervising)" as n_79341770723a
    [*] --> n_79341770723a
    n_79341770723a --> n_be98c9fd9851
    state "root.index.html (query)" as n_be98c9fd9851
    n_79341770723a --> n_e95c272b90c6
    state "root.styles.css (query)" as n_e95c272b90c6
    n_79341770723a --> n_a6ffd1e972f3
    state "root.main.js (query)" as n_a6ffd1e972f3
    n_79341770723a --> n_3f11510eb976
    state "root.flock.js (query)" as n_3f11510eb976
    n_79341770723a --> n_8f501d291960
    state "root.boid.js (query)" as n_8f501d291960
    n_79341770723a --> n_3e96bed24f6a
    state "root.utils.js (query)" as n_3e96bed24f6a
```

## 8. Full markdown report

`report_md` bundles tree + gantt + code log + token sparkline + error summary into a single Markdown document — perfect for pasting into a PR or a postmortem.

In [19]:
from rlmflow.utils.viz import report_md

Markdown(report_md(states, title="boids — coding-agent run"))

# boids — coding-agent run

**Steps:** 5
**Agents:** 7
**Tokens:** 43,708 (28,684 in, 15,024 out)
**Outcome:** result

## Tree

```
root [result] {default:gpt-5} -> Boids simulation created in output/boids-simulation with split files and trails.
  root.index.html [result] {fast:gpt-5-mini} -> ok
  root.styles.css [result] {fast:gpt-5-mini} -> ok
  root.main.js [result] {fast:gpt-5-mini} -> ok
  root.flock.js [result] {fast:gpt-5-mini} -> ok
  root.boid.js [result] {fast:gpt-5-mini} -> ok
  root.utils.js [result] {fast:gpt-5-mini} -> ok
```

## Cumulative tokens

```
 ▁▆██   43708 tok over 5 steps
```

## Result

```
Boids simulation created in output/boids-simulation with split files and trails. Open index.html to run.
```


## Next

- Full visualization tour (every helper, every format): [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb)
- Run the same agent from a terminal: `python examples/coding-agent/agent.py --workspace ./myproject`
- Render any saved trace from the CLI: `rlmflow render <trace-dir> -f mermaid-flowchart` / `report-md` / `gantt-html`
- Read the visualization roadmap: [`docs/internal/visualization_roadmap.md`](../../docs/internal/visualization_roadmap.md)